# Install Dependecies

In [1]:
%%capture
%pip install -q "nltk>=3.9,<4" "spacy>=3.8,<4" "transformers>=5,<6"
%pip install matplotlib
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [2]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [48]:
import ipynbname
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import time
from collections import Counter
from datasets import load_dataset
from pathlib import Path
from tqdm.std import tqdm
from transformers import AutoTokenizer


# Set `ROOT_DIR`

In [4]:
ROOT_DIR = ipynbname.path().parent
ROOT_DIR = Path(ROOT_DIR)
print(ROOT_DIR)

/home/tlvj/msc_datalogi/2_semester/nlp/msc-nlp-2026/project_notebooks


# Import datasets

In [ ]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

### Tokeniser

In [6]:
xlm_tokeniser = AutoTokenizer.from_pretrained("xlm-roberta-base")

### Pick languages

In [56]:
LANGUAGES = ["ar", "ko", "te"]

### Preprocessing

In [57]:
df_train = df_train[df_train["lang"].isin(LANGUAGES)].copy()
df_val = df_val[df_val["lang"].isin(LANGUAGES)].copy()

# 2 Week 36: Data and Rule-Based Baselines
Download the dataset and inspect its columns. Report Item 1 separately by
language and split, plus overall example counts and answerability proportions.
Compute Item 2 from the training questions separately by language. Apply
Item 3 to every answerable example in both splits.

3. verify programmatically that every answerable item’s answer equals the
substring beginning at answer start, and report the number checked and
any failures.

Implement and evaluate two answerability baselines: (i) the majority-class
baseline estimated from the training split and (ii) a deterministic rule-based
classifier that uses only the question and context. The rule may use tokenisation,
lexical features or machine translation, but no labelled validation examples or
trained answerability/QA model. Discuss what information the rule can and
cannot exploit in this cross-lingual setting.

## Week 36.1
Report the number of examples, answerable/unanswerable proportions, median and interquartile range of tokenised question and context lengths (a table is sufficient; plots are optional), missing values and exact duplicate question–context pairs.

In [58]:
def token_len(texts):
    return [len(xlm_tokeniser(t)["input_ids"]) for t in texts]

In [59]:
df_train["q_len"] = token_len(df_train["question"])
df_train["c_len"] = token_len(df_train["context"])
df_val["q_len"] = token_len(df_val["question"])
df_val["c_len"] = token_len(df_val["context"])

### Compute the number of answerable / unanswerable / pct_unanswerable

In [60]:
# train
counts = df_train.groupby(["lang"])["answerable"].agg(n="count", answerable="sum")
counts["unanswerable"] = counts["n"] - counts["answerable"]
counts["pct_unanswerable"] = (100 * counts["unanswerable"] / counts["n"]).round(0)
print(counts.to_markdown())

| lang   |    n |   answerable |   unanswerable |   pct_unanswerable |
|:-------|-----:|-------------:|---------------:|-------------------:|
| ar     | 2558 |         2303 |            255 |                 10 |
| ko     | 2422 |         2359 |             63 |                  3 |
| te     | 1355 |         1310 |             45 |                  3 |


In [61]:
# val
counts = df_val.groupby("lang")["answerable"].agg(n="count", answerable="sum")
counts["unanswerable"] = counts["n"] - counts["answerable"]
counts["pct_unanswerable"] = (100 * counts["unanswerable"] / counts["n"]).round(0)
print(counts.to_markdown())

| lang   |   n |   answerable |   unanswerable |   pct_unanswerable |
|:-------|----:|-------------:|---------------:|-------------------:|
| ar     | 415 |          363 |             52 |                 13 |
| ko     | 356 |          337 |             19 |                  5 |
| te     | 384 |          291 |             93 |                 24 |


### Overall counts

In [62]:
# train
n = len(df_train)
answerable = df_train["answerable"].sum()
print("n:", n, "| answerable:", answerable, "| unanswerable:", n - answerable,
      "| pct_unanswerable:", round(100 * (n - answerable) / n, 2))

n: 6335 | answerable: 5972 | unanswerable: 363 | pct_unanswerable: 5.73


In [63]:
# val
n = len(df_val)
answerable = df_val["answerable"].sum()
print("n:", n, "| answerable:", answerable, "| unanswerable:", n - answerable,
      "| pct_unanswerable:", round(100 * (n - answerable) / n, 2))

n: 1155 | answerable: 991 | unanswerable: 164 | pct_unanswerable: 14.2


### Median and IQR of question / context length

In [69]:
# train
grouped = df_train.groupby("lang")
lengths = pd.DataFrame({
    "q_q25": grouped["q_len"].quantile(0.25),
    "q_median": grouped["q_len"].median(),
    "q_q75": grouped["q_len"].quantile(0.75),
    "c_q25": grouped["c_len"].quantile(0.25),
    "c_median": grouped["c_len"].median(),
    "c_q75": grouped["c_len"].quantile(0.75),
})
lengths["q_iqr"] = lengths["q_q75"] - lengths["q_q25"]
lengths["c_iqr"] = lengths["c_q75"] - lengths["c_q25"]
print(lengths.to_markdown())

| lang   |   q_q25 |   q_median |   q_q75 |   c_q25 |   c_median |   c_q75 |   q_iqr |   c_iqr |
|:-------|--------:|-----------:|--------:|--------:|-----------:|--------:|--------:|--------:|
| ar     |      11 |         13 |      15 |      89 |        133 |     191 |       4 |     102 |
| ko     |      12 |         14 |      16 |      85 |        127 |     181 |       4 |      96 |
| te     |      11 |         13 |      16 |      80 |        121 |     171 |       5 |      91 |


In [70]:
# val
grouped = df_val.groupby("lang")
lengths = pd.DataFrame({
    "q_q25": grouped["q_len"].quantile(0.25),
    "q_median": grouped["q_len"].median(),
    "q_q75": grouped["q_len"].quantile(0.75),
    "c_q25": grouped["c_len"].quantile(0.25),
    "c_median": grouped["c_len"].median(),
    "c_q75": grouped["c_len"].quantile(0.75),
})
lengths["q_iqr"] = lengths["q_q75"] - lengths["q_q25"]
lengths["c_iqr"] = lengths["c_q75"] - lengths["c_q25"]
print(lengths.to_markdown())

| lang   |   q_q25 |   q_median |   q_q75 |   c_q25 |   c_median |   c_q75 |   q_iqr |   c_iqr |
|:-------|--------:|-----------:|--------:|--------:|-----------:|--------:|--------:|--------:|
| ar     |      11 |         12 |      15 |   89.5  |      131   |   190.5 |       4 |  101    |
| ko     |      12 |         14 |      16 |   80.75 |      123.5 |   183.5 |       4 |  102.75 |
| te     |      12 |         14 |      18 |  114    |      154   |   208   |       6 |   94    |


### Missing values

In [66]:
print(df_train.isna().sum().to_markdown())
print("-------------------------")
print(df_val.isna().sum().to_markdown())

|               |    0 |
|:--------------|-----:|
| question      |    0 |
| context       |    0 |
| lang          |    0 |
| answerable    |    0 |
| answer_start  |    0 |
| answer        |    0 |
| answer_inlang | 6285 |
| q_len         |    0 |
| c_len         |    0 |
-------------------------
|               |    0 |
|:--------------|-----:|
| question      |    0 |
| context       |    0 |
| lang          |    0 |
| answerable    |    0 |
| answer_start  |    0 |
| answer        |    0 |
| answer_inlang | 1055 |
| q_len         |    0 |
| c_len         |    0 |


### Duplicate pairs

In [67]:
print("train:", df_train.duplicated(subset=["question", "context"]).sum())
print("val:", df_val.duplicated(subset=["question", "context"]).sum())

train: 10
val: 0


## Week 36.2
Report the five most common question tokens and their counts for each language, together with an English translation, and explain your tokenisation.

In [72]:
PUNCT = set("?؟.,!:;\"'()،")

In [71]:
def xlm_tokens(text):
    return [t.lstrip("▁") for t in xlm_tokeniser.tokenize(text)]

In [73]:
def word_tokens(text):
    return [t.strip("".join(PUNCT)) for t in text.split()]

In [74]:
def top_tokens(texts, tokenise, n=5):
    counts = Counter()
    for text in texts:
        counts.update(t for t in tokenise(text) if t and t not in PUNCT)
    return counts.most_common(n)

In [79]:
def top_token_table(tokenise, n=5):
    rows = []
    for lang in LANGUAGES:
        questions = df_train.loc[df_train["lang"] == lang, "question"]
        for token, count in top_tokens(questions, tokenise, n):
            rows.append({"lang": lang, "token": token, "count": count})
    return pd.DataFrame(rows)

table = top_token_table(xlm_tokens)
table

,lang,token,count
0,ar,م,720
1,ar,في,660
2,ar,من,606
3,ar,تى,536
4,ar,ما,478
5,ko,는,1154
6,ko,은,985
7,ko,가,707
8,ko,의,606
9,ko,인가,599


In [ ]:
# the table above was given to an ai tool, to translate the tokens to english
"""Prompt: Translate these tokens in this table to english"""
TRANSLATIONS = {
    # arabic
    "م": "the letter mīm — a fragment, not a word (e.g. from كم 'how many', or a prefix)",
    "في": "in",
    "من": "from / of",
    "تى": "fragment ending in -tā — most likely the tail of متى 'when' (or حتى 'until' / التى 'which')",
    "ما": "'what' (also the negator 'not')",
    # korean
    "는": "topic particle ('as for…') — no English equivalent",
    "은": "topic particle (same as 는, after a consonant)",
    "가": "subject particle",
    "의": "possessive particle ('of / 's')",
    "인가": "'is it…?' — copula + question ending",
    # telugu
    "లో": "in / inside",
    "ఎవరు": "who",
    "ఏది": "which / which one",
    "౦౦": "'00' — two Telugu-script zero digits",
    "ఎన్ని": "how many"
}

In [80]:
table["english"] = table["token"].map(TRANSLATIONS).fillna("?")
table

,lang,token,count,english
0,ar,م,720,fragment of متى (when)
1,ar,في,660,in
2,ar,من,606,from / who
3,ar,تى,536,fragment of متى (when)
4,ar,ما,478,what
5,ko,는,1154,topic particle
6,ko,은,985,topic particle
7,ko,가,707,subject particle
8,ko,의,606,genitive 'of'
9,ko,인가,599,is it? (copula + question ending)
